# Phase 2 grounding: GRL-B's released denoising outputs, rescored on our test-set copies (CPU, no network run)

GRL (CVPR 2023) released its denoising outputs (`visual_results_dn.tar.gz`). Rescoring them against other copies of
the clean test sets measures how much of a published PSNR depends on the copy and the scoring protocol alone, with
the denoiser held fixed. The copies used here are the ones our runs used: colour CBSD68 / Kodak24 / McMaster /
Urban100 from DMID's release, and gray Set12 / BSD68 from the DnCNN repository. Gray Urban100 is the BT.601 luma of
DMID's colour copy, rounded, as `cv2.imread(path, 0)` gives.

Outputs: `outputs/rescored.csv` (one row per output image), `outputs/summary.csv` (means per set, σ and folder),
`outputs/tree.txt` (the archive layout).
Kaggle: CPU, internet on.

In [ ]:
import io, os, re, tarfile, time, urllib.request, zipfile
import numpy as np
import pandas as pd
from PIL import Image

OUT, CACHE = "outputs", "/tmp/cache"
for d in (OUT, CACHE):
    os.makedirs(d, exist_ok=True)
T0 = time.time()


def download(url, path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        for attempt in range(5):
            try:
                with urllib.request.urlopen(url, timeout=300) as r, open(path, "wb") as f:
                    while chunk := r.read(1 << 23):
                        f.write(chunk)
                return path
            except Exception as e:
                print(f"  retry {attempt + 1}: {e}", flush=True)
                time.sleep(10 * (attempt + 1))
        raise RuntimeError(url)
    return path


tgz = download("https://github.com/ofsoundof/GRL-Image-Restoration/releases/download/v1.0.0/visual_results_dn.tar.gz",
               f"{CACHE}/visual_results_dn.tar.gz")
print(f"archive {os.path.getsize(tgz) / 1e9:.2f} GB ({(time.time() - T0) / 60:.1f} min)", flush=True)
ROOT = f"{CACHE}/grl"
if not os.path.isdir(ROOT):
    with tarfile.open(tgz) as t:
        t.extractall(ROOT)
files = sorted(os.path.join(dp, f) for dp, _, fs in os.walk(ROOT) for f in fs if f.lower().endswith((".png", ".tif", ".bmp", ".jpg")))
dirs = sorted({os.path.relpath(os.path.dirname(f), ROOT) for f in files})
with open(f"{OUT}/tree.txt", "w") as f:
    for d in dirs:
        n = sum(1 for x in files if os.path.dirname(x) == os.path.join(ROOT, d))
        f.write(f"{n:5d}  {d}\n")
print(len(files), "images in", len(dirs), "folders")
print(open(f"{OUT}/tree.txt").read()[:4000])
print("examples:", [os.path.relpath(x, ROOT) for x in files[:5]])

## Clean copies

In [ ]:
def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


zpath = download("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", f"{CACHE}/Datasets.zip")
CLEAN = {}
with zipfile.ZipFile(zpath) as z:
    for ds in ("CBSD68", "Kodak24", "McMaster", "Urban100"):
        names = sorted([n for n in z.namelist() if n.startswith(f"Datasets/Gaussion/{ds}/") and not n.endswith("/")], key=natkey)
        CLEAN[("color", ds)] = {os.path.splitext(n.split("/")[-1])[0]: np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB")) for n in names}
RAW = "https://raw.githubusercontent.com/cszn/DnCNN/master/testsets/{}/{}"
CLEAN[("gray", "Set12")] = {f"{i:02d}": np.asarray(Image.open(download(RAW.format("Set12", f"{i:02d}.png"), f"{CACHE}/Set12_{i:02d}.png")).convert("L"))
                            for i in range(1, 13)}
CLEAN[("gray", "BSD68")] = {f"test{i:03d}": np.asarray(Image.open(download(RAW.format("BSD68", f"test{i:03d}.png"), f"{CACHE}/BSD68_{i:03d}.png")).convert("L"))
                            for i in range(1, 69)}


def luma601(rgb):
    """cv2.imread(path, 0): Y = 0.299 R + 0.587 G + 0.114 B, rounded."""
    return np.clip(np.round(rgb.astype(np.float64) @ np.array([0.299, 0.587, 0.114])), 0, 255).astype(np.uint8)


CLEAN[("gray", "Urban100")] = {k: luma601(v) for k, v in CLEAN[("color", "Urban100")].items()}
for k, v in CLEAN.items():
    print(k, len(v), "clean images; e.g.", list(v)[:3])

## Match every output to a clean image by content, with the crop offset, and rescore on the overlap

File names differ between copies (DMID renamed CBSD68 to `0-1 …`, Urban100 is 0-indexed), so each output is matched
to the clean image and crop offset that minimise the MSE. Offsets tried: top-left, centre, bottom-right and the
1-px border crop. A transposed clean image is also tried. PSNR is computed on the overlap only.

In [ ]:
SETS = {"cbsd68": "CBSD68", "kodak24": "Kodak24", "mcmaster": "McMaster", "urban100": "Urban100", "set12": "Set12",
        "bsd68": "BSD68"}


def parse(path):
    rel = os.path.relpath(path, ROOT)
    parts = rel.split(os.sep)
    ds = next((SETS[p.lower()] for p in parts if p.lower() in SETS), None)
    m = re.search(r"Sigma(15|25|50)", rel)
    model = parts[1] if len(parts) > 2 else ""
    return rel, ds, int(m.group(1)) if m else None, model


def offsets(H, W, h, w):
    cand = {(0, 0), ((H - h) // 2, (W - w) // 2), (H - h, W - w), (min(1, H - h), min(1, W - w))}
    return [(i, j) for i, j in cand if 0 <= i <= H - h and 0 <= j <= W - w]


def best_match(out, pool):
    h, w = out.shape[:2]
    best = (np.inf, None, None, False)
    o = out.astype(np.float64)
    for key, clean in pool.items():
        for transposed in (False, True):
            c = np.swapaxes(clean, 0, 1) if transposed else clean
            H, W = c.shape[:2]
            if H < h or W < w:
                continue
            for i, j in offsets(H, W, h, w):
                mse = np.mean((o - c[i:i + h, j:j + w].astype(np.float64)) ** 2)
                if mse < best[0]:
                    best = (mse, key, (i, j), transposed)
    return best


rows = []
for path in files:
    rel, ds, sigma, model = parse(path)
    if ds is None:
        continue
    kind = "gray" if "gray" in model else "color"
    out = np.asarray(Image.open(path).convert("L" if kind == "gray" else "RGB"))
    mse, key, off, transposed = best_match(out, CLEAN[(kind, ds)])
    if key is None:
        rows.append(dict(file=rel, model=model, dataset=ds, kind=kind, sigma=sigma, clean=None, psnr=np.nan))
        continue
    H, W = CLEAN[(kind, ds)][key].shape[:2]
    rows.append(dict(file=rel, model=model, dataset=ds, kind=kind, sigma=sigma, clean=key, out_h=out.shape[0], out_w=out.shape[1],
                     clean_h=H, clean_w=W, off_i=off[0], off_j=off[1], transposed=transposed,
                     coverage=out.shape[0] * out.shape[1] / (H * W), psnr=10 * np.log10(255 ** 2 / mse) if mse > 0 else np.inf))
R = pd.DataFrame(rows)
R.to_csv(f"{OUT}/rescored.csv", index=False)
S = (R.dropna(subset=["psnr"]).groupby(["kind", "dataset", "model", "sigma"])
     .agg(n=("psnr", "size"), psnr=("psnr", "mean"), min_psnr=("psnr", "min"), coverage=("coverage", "mean"),
          distinct_clean=("clean", "nunique")).reset_index())
S.to_csv(f"{OUT}/summary.csv", index=False)
print(S.round(3).to_string(index=False))
R["out_shape"] = R.out_h.astype("Int64").astype(str) + "x" + R.out_w.astype("Int64").astype(str)
R["clean_shape"] = R.clean_h.astype("Int64").astype(str) + "x" + R.clean_w.astype("Int64").astype(str)
R["offset"] = "(" + R.off_i.astype("Int64").astype(str) + "," + R.off_j.astype("Int64").astype(str) + ")"
for col in ("out_shape", "clean_shape", "offset"):
    print(col, R.groupby("dataset")[col].agg(lambda s: s.value_counts().head(4).to_dict()).to_string(), sep="\n")
print("mean coverage of the clean image:", R.groupby("dataset").coverage.mean().round(3).to_dict())
print(f"done in {(time.time() - T0) / 60:.1f} min")